# 🪪 Notebook 3: OAuth 2.0 Authorization Code Flow

**The problem:** the user wants to let *AcmePhoto* see their Google Photos — **without** giving AcmePhoto their Google password.

OAuth solves this with a careful dance between four roles:

- **Resource Owner** — the user.
- **Client** — the app that wants access (AcmePhoto).
- **Authorization Server** — issues access tokens (Google's login screen).
- **Resource Server** — holds the data (the Google Photos API).

We will simulate the **authorization code** flow with two tiny FastAPI apps: one is the auth server, one is the client app. We will run them in the notebook with `httpx` for the HTTP calls.

> ⚠️ **OAuth 2.0 is not a login protocol.** It answers *"what may this app do on my
> behalf?"* (**authorization**), not *"who is this user?"* (**authentication**). An
> access token is a valet key, not an ID card — it says nothing verifiable about who
> the user is. The layer that adds authentication is **OpenID Connect**, covered at
> the end. Treating a raw OAuth access token as proof of identity is a classic and
> exploitable mistake.

## Learning objectives
- See the *redirect → consent → code → token → API call* sequence step by step.
- Understand why we exchange a *code* for a *token* (instead of returning the token directly).
- Watch the protections do their job: `state`, redirect-URI allow-listing, single-use
  codes, and **PKCE** — each demonstrated by an attack that fails.
- Tell OAuth (authorization) from OIDC (authentication).

## 🛠️ Setup

```bash
cd 01-foundations/authentication-authorization
uv sync
```

This is pure Python; no external services. We use FastAPI's in-process test client so we don't have to start real servers.

In [ ]:
from fastapi import FastAPI, HTTPException, Header
from fastapi.testclient import TestClient
import secrets, time, hashlib, base64

# -----------------------------------------------------------------------------
# Authorization Server (think: Google)
# -----------------------------------------------------------------------------
auth_server = FastAPI()

USERS = {"alice": "hunter2"}

# Clients register their redirect URIs and scopes up front. A *confidential*
# client (a web backend) also gets a secret; a *public* client (mobile app, SPA)
# cannot keep one, so it must use PKCE instead.
CLIENTS = {
    "acmephoto": {                                   # confidential
        "secret": "client-secret",
        "redirect_uris": {"https://acmephoto.example/callback"},
        "allowed_scopes": {"photos:read", "photos:write"},
    },
    "acmephoto-mobile": {                            # public: no secret, PKCE required
        "secret": None,
        "redirect_uris": {"com.acmephoto.app://callback"},
        "allowed_scopes": {"photos:read"},
        "require_pkce": True,
    },
}

CODES = {}           # code -> {user, client_id, scopes, redirect_uri, code_challenge}
TOKENS = {}          # access_token -> {user, scopes, exp}
REFRESH_TOKENS = {}  # refresh_token -> {user, client_id, scopes}

ACCESS_TTL = 2  # seconds — short so the demo can show expiry quickly


def _issue_tokens(user: str, client_id: str, scopes: set[str]):
    access = secrets.token_urlsafe(16)
    refresh = secrets.token_urlsafe(24)
    TOKENS[access] = {"user": user, "scopes": scopes, "exp": time.time() + ACCESS_TTL}
    REFRESH_TOKENS[refresh] = {"user": user, "client_id": client_id, "scopes": scopes}
    return access, refresh


@auth_server.post("/authorize")
def authorize(username: str, password: str, client_id: str,
              redirect_uri: str, scope: str = "", state: str = "",
              code_challenge: str = "", code_challenge_method: str = ""):
    # ⚠️ Real flows never take credentials as parameters like this — the user types
    # them into the auth server's own browser page, and a password in a query string
    # ends up in access logs and browser history. This is a notebook shortcut.
    if USERS.get(username) != password:
        raise HTTPException(401, "bad creds")
    client = CLIENTS.get(client_id)
    if not client:
        raise HTTPException(400, "unknown client")
    # Reject redirect URIs that weren't pre-registered — this is critical. Without
    # it, an attacker sends the user to /authorize with their own redirect_uri and
    # the authorization code is delivered straight to them.
    if redirect_uri not in client["redirect_uris"]:
        raise HTTPException(400, "bad redirect_uri")
    # Only allow scopes the client registered for.
    requested = set(scope.split()) if scope else set()
    if not requested.issubset(client["allowed_scopes"]):
        raise HTTPException(400, "scope not allowed")
    # Public clients have no secret, so PKCE is the only thing binding the code
    # to the app that started the flow.
    if client.get("require_pkce"):
        if not code_challenge or code_challenge_method != "S256":
            raise HTTPException(400, "PKCE required (S256)")

    code = secrets.token_urlsafe(8)
    CODES[code] = {
        "user": username,
        "client_id": client_id,
        "scopes": requested,
        "redirect_uri": redirect_uri,
        "code_challenge": code_challenge or None,
    }
    # The `state` parameter is echoed back verbatim — the client uses it to
    # detect CSRF (it must match what the client sent originally).
    return {"redirect_to": f"{redirect_uri}?code={code}&state={state}"}


@auth_server.post("/token")
def token(grant_type: str = "authorization_code",
          code: str = "", client_id: str = "", client_secret: str = "",
          redirect_uri: str = "", refresh_token: str = "",
          code_verifier: str = ""):
    client = CLIENTS.get(client_id)
    if not client:
        raise HTTPException(401, "bad client")
    # Confidential clients prove who they are with a secret; public ones can't.
    if client["secret"] is not None and client["secret"] != client_secret:
        raise HTTPException(401, "bad client")

    if grant_type == "authorization_code":
        # CODES.pop => an authorization code is single-use. Replaying it fails.
        info = CODES.pop(code, None)
        if not info or info["client_id"] != client_id:
            raise HTTPException(400, "bad code")
        # redirect_uri must match the one used in /authorize.
        if info["redirect_uri"] != redirect_uri:
            raise HTTPException(400, "redirect_uri mismatch")
        # PKCE: the verifier must hash to the challenge sent at /authorize.
        if info["code_challenge"]:
            if not code_verifier:
                raise HTTPException(400, "code_verifier required")
            digest = hashlib.sha256(code_verifier.encode()).digest()
            computed = base64.urlsafe_b64encode(digest).rstrip(b"=").decode()
            if not secrets.compare_digest(computed, info["code_challenge"]):
                raise HTTPException(400, "PKCE verification failed")
        access, refresh = _issue_tokens(info["user"], client_id, info["scopes"])
        return {"access_token": access, "refresh_token": refresh,
                "expires_in": ACCESS_TTL, "scope": " ".join(sorted(info["scopes"]))}

    if grant_type == "refresh_token":
        info = REFRESH_TOKENS.get(refresh_token)
        if not info or info["client_id"] != client_id:
            raise HTTPException(400, "bad refresh")
        access, _ = _issue_tokens(info["user"], client_id, info["scopes"])
        return {"access_token": access, "expires_in": ACCESS_TTL,
                "scope": " ".join(sorted(info["scopes"]))}

    raise HTTPException(400, "unsupported grant_type")


# -----------------------------------------------------------------------------
# Resource Server (think: Google Photos API)
# -----------------------------------------------------------------------------
resource_server = FastAPI()

@resource_server.get("/photos")
def photos(authorization: str = Header(default="")):
    # RFC 6750: the token goes in the Authorization header, NOT in the query
    # string. Query strings land in access logs, browser history, and `Referer`
    # headers sent to third parties — all places a bearer token must never be.
    scheme, _, access_token = authorization.partition(" ")
    if scheme.lower() != "bearer" or not access_token:
        raise HTTPException(401, "missing bearer token")
    info = TOKENS.get(access_token)
    if not info:
        raise HTTPException(401, "bad token")
    if info["exp"] < time.time():
        raise HTTPException(401, "token expired")
    if "photos:read" not in info["scopes"]:
        raise HTTPException(403, "missing scope photos:read")
    return {"user": info["user"], "photos": ["beach.jpg", "cat.jpg"]}


def bearer(access_token: str) -> dict:
    """Helper: the header every API call in this notebook sends."""
    return {"Authorization": f"Bearer {access_token}"}


auth_client = TestClient(auth_server)
res_client = TestClient(resource_server)
print("FastAPI apps ready ✅ (state, scopes, expiry, refresh, PKCE, bearer auth)")

In [ ]:
# -----------------------------------------------------------------------------
# THE FLOW (annotated step by step)
# -----------------------------------------------------------------------------

# The client generates a random `state` BEFORE redirecting. It remembers it
# locally (e.g. in the user's session) so it can verify the callback later.
client_state = secrets.token_urlsafe(8)
print("Client generated state:", client_state)

# Step 1: AcmePhoto sends Alice to the auth server with client_id, redirect_uri,
# scope, and state. Alice logs in there and clicks "Allow".
r = auth_client.post("/authorize", params={
    "username": "alice", "password": "hunter2",
    "client_id": "acmephoto",
    "redirect_uri": "https://acmephoto.example/callback",
    "scope": "photos:read",
    "state": client_state,
})
redirect = r.json()["redirect_to"]
print("Step 1 — auth server redirects browser to:", redirect)

# Step 2: AcmePhoto's backend parses the callback URL.
from urllib.parse import urlparse, parse_qs
qs = parse_qs(urlparse(redirect).query)
code = qs["code"][0]
returned_state = qs["state"][0]

# CRITICAL: verify the state matches what we sent. If not, this could be CSRF.
assert returned_state == client_state, "state mismatch — possible CSRF!"
print("Step 2 — state verified ✅, received code:", code)

# Step 3: AcmePhoto exchanges the code for tokens, proving its identity with
# client_secret. This is a server-to-server back-channel call.
r = auth_client.post("/token", params={
    "grant_type": "authorization_code",
    "code": code,
    "client_id": "acmephoto",
    "client_secret": "client-secret",
    "redirect_uri": "https://acmephoto.example/callback",
})
body = r.json()
access_token = body["access_token"]
refresh_token = body["refresh_token"]
print("Step 3 — got tokens:", body)

# Step 4: AcmePhoto calls the Resource Server with the access token, in the
# Authorization header (never in the URL).
r = res_client.get("/photos", headers=bearer(access_token))
print("Step 4 — photos:", r.json())
assert r.status_code == 200 and r.json()["user"] == "alice"

# A request with no token at all must be refused.
assert res_client.get("/photos").status_code == 401


## 🛡️ Now break it — three attacks that should fail

A protocol description is easy to nod along to. The useful exercise is to *play the
attacker* and watch each guard actually stop you.

In [ ]:
# --- Attack 1: open redirect. Send the user to the real auth server, but ask for
# the code to be delivered to an attacker-controlled URL. ---
r = auth_client.post("/authorize", params={
    "username": "alice", "password": "hunter2",
    "client_id": "acmephoto",
    "redirect_uri": "https://evil.example/steal",   # not registered
    "scope": "photos:read",
    "state": secrets.token_urlsafe(8),
})
print("unregistered redirect_uri →", r.status_code, r.json())
assert r.status_code == 400 and r.json()["detail"] == "bad redirect_uri"

# Sub-domain and path tricks must fail too — the allow-list is an exact match, not
# a prefix match. "startswith" checks here are a well-known source of takeovers.
for sneaky in ("https://acmephoto.example.evil.com/callback",
               "https://acmephoto.example/callback/../../evil",
               "https://acmephoto.example/callback?next=https://evil.example"):
    r = auth_client.post("/authorize", params={
        "username": "alice", "password": "hunter2", "client_id": "acmephoto",
        "redirect_uri": sneaky, "scope": "photos:read", "state": "x"})
    assert r.status_code == 400, (sneaky, r.status_code)
print("...and three near-miss variations were rejected too ✅")

In [ ]:
# --- Attack 2: replay a used authorization code. ---
# Codes are single-use: the server pops them on redemption.
r = auth_client.post("/authorize", params={
    "username": "alice", "password": "hunter2", "client_id": "acmephoto",
    "redirect_uri": "https://acmephoto.example/callback",
    "scope": "photos:read", "state": "s1"})
one_time_code = parse_qs(urlparse(r.json()["redirect_to"]).query)["code"][0]

exchange = lambda: auth_client.post("/token", params={
    "grant_type": "authorization_code", "code": one_time_code,
    "client_id": "acmephoto", "client_secret": "client-secret",
    "redirect_uri": "https://acmephoto.example/callback"})

first, second = exchange(), exchange()
print("first exchange :", first.status_code)
print("second exchange:", second.status_code, second.json())
assert first.status_code == 200
assert second.status_code == 400, "authorization codes must be single-use"

In [ ]:
# --- Attack 3: wrong client tries to redeem someone else's code. ---
r = auth_client.post("/authorize", params={
    "username": "alice", "password": "hunter2", "client_id": "acmephoto",
    "redirect_uri": "https://acmephoto.example/callback",
    "scope": "photos:read", "state": "s2"})
victim_code = parse_qs(urlparse(r.json()["redirect_to"]).query)["code"][0]

# The attacker knows the code but not acmephoto's secret.
r = auth_client.post("/token", params={
    "grant_type": "authorization_code", "code": victim_code,
    "client_id": "acmephoto", "client_secret": "guessed-wrong",
    "redirect_uri": "https://acmephoto.example/callback"})
print("wrong client_secret →", r.status_code, r.json())
assert r.status_code == 401

# And redeeming with a redirect_uri that differs from the one used at /authorize
# fails, even with the right secret — that binding is what stops a code obtained
# through one redirect from being spent against another.
r = auth_client.post("/token", params={
    "grant_type": "authorization_code", "code": victim_code,
    "client_id": "acmephoto", "client_secret": "client-secret",
    "redirect_uri": "https://acmephoto.example/other"})
print("redirect_uri mismatch →", r.status_code, r.json())
assert r.status_code == 400

## 🎯 Scopes: principle of least privilege

A scope is a label on the token that says *what it is allowed to do*. `photos:read` lets you list photos; `photos:write` would let you upload. The resource server checks the scope on every request.

Let's see what happens if AcmePhoto asks for a scope the client isn't registered for, and what happens if it tries to hit an endpoint the token doesn't cover.

In [ ]:
# 1) Requesting a scope that is not registered for this client → rejected.
r = auth_client.post("/authorize", params={
    "username": "alice", "password": "hunter2",
    "client_id": "acmephoto",
    "redirect_uri": "https://acmephoto.example/callback",
    "scope": "admin:everything",   # not in allowed_scopes
    "state": secrets.token_urlsafe(8),
})
print("unregistered scope →", r.status_code, r.json())
assert r.status_code == 400

# 2) Requesting a scope but then calling an endpoint that requires a DIFFERENT scope.
#    For this demo we only have /photos which requires photos:read, so asking for
#    photos:write alone should fail.
r = auth_client.post("/authorize", params={
    "username": "alice", "password": "hunter2",
    "client_id": "acmephoto",
    "redirect_uri": "https://acmephoto.example/callback",
    "scope": "photos:write",
    "state": "x",
})
code2 = parse_qs(urlparse(r.json()["redirect_to"]).query)["code"][0]
r = auth_client.post("/token", params={
    "grant_type": "authorization_code", "code": code2,
    "client_id": "acmephoto", "client_secret": "client-secret",
    "redirect_uri": "https://acmephoto.example/callback",
})
narrow_token = r.json()["access_token"]
r = res_client.get("/photos", headers=bearer(narrow_token))
print("wrong-scope request →", r.status_code, r.json())
assert r.status_code == 403, "a photos:write token must not read photos"


## ⏳ Token expiry and the `refresh_token`

Access tokens are deliberately **short-lived** (we set 2 seconds — production is typically 5–60 minutes). When one expires, the client exchanges a **refresh token** for a new access token, *without* bothering the user again. Refresh tokens live longer and are stored only on the client's backend.

Why two tokens? If an access token leaks, the damage window is tiny. Refresh tokens don't fly out on every API call, so they leak far less often.

In [ ]:
# Right now the token is fresh.
r = res_client.get("/photos", headers=bearer(access_token))
print("Calling /photos right now:", r.status_code, r.json())
assert r.status_code == 200

time.sleep(ACCESS_TTL + 0.5)
r = res_client.get("/photos", headers=bearer(access_token))
print("After expiry      :", r.status_code, r.json())
assert r.status_code == 401 and r.json()["detail"] == "token expired"

# Use the refresh_token to get a fresh access_token — no user interaction needed.
r = auth_client.post("/token", params={
    "grant_type": "refresh_token",
    "refresh_token": refresh_token,
    "client_id": "acmephoto",
    "client_secret": "client-secret",
})
new_access = r.json()["access_token"]
r = res_client.get("/photos", headers=bearer(new_access))
print("Refreshed         :", r.status_code, r.json())
assert r.status_code == 200
assert new_access != access_token, "a refresh must mint a NEW access token"

## 🔑 PKCE — the fix for clients that can't keep a secret

Everything above leaned on `client_secret` to prove that the app redeeming the code is
the app that started the flow. A mobile app or SPA **cannot hold a secret** — anyone can
unzip the bundle and read it. So what stops an attacker who intercepts the authorization
code (via a malicious app registered for the same custom URL scheme, a leaked log, or
the browser history) from redeeming it themselves?

**PKCE** ("pixie", *Proof Key for Code Exchange*, RFC 7636) replaces the static secret
with a fresh one-time secret per login:

1. Before redirecting, the client generates a random **`code_verifier`**.
2. It sends only `code_challenge = BASE64URL(SHA256(verifier))` to `/authorize`.
   The challenge travels through the browser, where an attacker may see it — that's
   fine, it's a hash.
3. At `/token` the client presents the raw **`code_verifier`**. The server hashes it
   and compares. Only the app that generated the verifier can complete the exchange.

A stolen code is now worthless on its own. **RFC 9700 (OAuth 2.0 Security Best Current
Practice) requires PKCE for *all* clients**, confidential ones included — it also
defends against code injection, which a client secret does not.

In [ ]:
import hashlib, base64, secrets

def pkce_pair():
    """Generate (code_verifier, code_challenge) using the S256 method."""
    verifier = secrets.token_urlsafe(64)[:128]          # RFC 7636: 43-128 chars
    digest = hashlib.sha256(verifier.encode()).digest()
    challenge = base64.urlsafe_b64encode(digest).rstrip(b"=").decode()
    return verifier, challenge


def start_mobile_flow():
    """Run the front-channel half of a PKCE login; return (verifier, code)."""
    verifier, challenge = pkce_pair()
    state = secrets.token_urlsafe(8)
    r = auth_client.post("/authorize", params={
        "username": "alice", "password": "hunter2",
        "client_id": "acmephoto-mobile",
        "redirect_uri": "com.acmephoto.app://callback",
        "scope": "photos:read",
        "state": state,
        "code_challenge": challenge,
        "code_challenge_method": "S256",
    })
    assert r.status_code == 200, r.json()
    qs = parse_qs(urlparse(r.json()["redirect_to"]).query)
    assert qs["state"][0] == state
    return verifier, qs["code"][0]


verifier, challenge = pkce_pair()
print(f"code_verifier  ({len(verifier)} chars): {verifier[:32]}...")
print(f"code_challenge (S256)      : {challenge}")
print("\nOnly the challenge is sent through the browser. The verifier never leaves")
print("the device until the back-channel /token call.")

In [ ]:
def redeem(code, **extra):
    return auth_client.post("/token", params={
        "grant_type": "authorization_code", "code": code,
        "client_id": "acmephoto-mobile",
        "redirect_uri": "com.acmephoto.app://callback",
        **extra,
    })

# --- The attack PKCE exists to stop ---
# A malicious app registered for the same custom URL scheme intercepts the code.
# It has everything the real app has except the verifier.

_, stolen = start_mobile_flow()
r = redeem(stolen)                       # no verifier at all
print("attacker, no verifier      →", r.status_code, r.json())
assert r.status_code == 400 and "verifier" in r.json()["detail"]

_, stolen = start_mobile_flow()          # a fresh code: see the note below
r = redeem(stolen, code_verifier=secrets.token_urlsafe(64)[:128])
print("attacker, guessed verifier →", r.status_code, r.json())
assert r.status_code == 400 and "PKCE" in r.json()["detail"]

# --- The real app, holding the verifier, completes the exchange ---
verifier, code = start_mobile_flow()
r = redeem(code, code_verifier=verifier)
print("real app, real verifier    →", r.status_code)
assert r.status_code == 200
mobile_token = r.json()["access_token"]
assert res_client.get("/photos", headers=bearer(mobile_token)).status_code == 200
print("\n✅ No client secret anywhere, and an intercepted code was still useless.")

# A public client must not be able to opt out of PKCE by simply omitting it.
r = auth_client.post("/authorize", params={
    "username": "alice", "password": "hunter2",
    "client_id": "acmephoto-mobile",
    "redirect_uri": "com.acmephoto.app://callback",
    "scope": "photos:read", "state": "x",
})
print("public client without PKCE →", r.status_code, r.json())
assert r.status_code == 400

> **Why a fresh code for each attempt?** The server pops the code *before* checking
> PKCE, so a failed exchange burns it — retrying the same code returns `bad code`
> rather than `PKCE verification failed`. That is deliberate: RFC 7636 says an
> authorization server SHOULD revoke a code whose verifier check fails, because a
> failed check means somebody other than the legitimate client has the code.

## ✅ Recap

The OAuth dance trades a tiny bit of complexity for a huge security win: **third-party
apps can act on behalf of the user without ever seeing the password**, and the user can
revoke access at any time.

Every protection in this notebook was demonstrated by an attack that failed:

| Protection | Stops | Demonstrated by |
|---|---|---|
| **`state`** | CSRF — an attacker completing a flow in your session | asserting the echoed value matches |
| **Redirect-URI allow-list** (exact match) | the code being delivered to the attacker | three rejected near-miss URIs |
| **Single-use codes** | replay of an intercepted code | second exchange returns 400 |
| **`client_secret`** | a different app redeeming your code | wrong secret returns 401 |
| **`redirect_uri` binding at `/token`** | a code obtained via one redirect being spent on another | mismatch returns 400 |
| **PKCE** | code interception where no secret can be kept | attacker without the verifier returns 400 |
| **Scopes** | an over-privileged token | `photos:write` token gets 403 on read |
| **Short access TTL + refresh** | a leaked token staying useful | 401 after 2 s |

### Things this demo simplifies

- Credentials are passed as parameters. Real flows put the login form **on the auth
  server**, and a password in a query string ends up in logs and history.
- Tokens here are opaque random strings looked up in a dict. Many providers issue JWTs
  instead — same flow, but the resource server verifies a signature rather than doing a
  lookup (see Notebook 2 for how to verify one properly).
- **Refresh-token rotation** is missing: production servers issue a *new* refresh token
  on every use and revoke the whole family if an old one reappears, which detects theft.
- There is no consent screen, no user-facing revocation, and no token introspection
  endpoint.

### 🆔 OAuth vs OpenID Connect

OAuth 2.0 is **authorization** — "what may this app do on my behalf?". The access token
is a capability; it is not addressed to you, may be opaque, and carries no verifiable
statement about the user's identity.

**OpenID Connect (OIDC)** is a thin layer on top that adds **authentication**. It
returns an extra **`id_token`**: a signed JWT, issued *to your client* (`aud` = your
`client_id`), with `iss`, `sub`, `exp` and a `nonce` you must check. "Log in with
Google" buttons are OIDC.

The mistake to avoid: **do not treat an access token as a login.** Calling a `/userinfo`
endpoint with someone's access token tells you which account the token belongs to, but
a token obtained by a *different* app for a *different* purpose will answer just as
happily — which is the "confused deputy" problem OIDC's audience-bound `id_token`
exists to solve.